## Phase 2: Build Multivariate Time-Series Dataset

We'll create two datasets:
 - **panel_quarterly_org.csv**: one row per (organization_id, quarter)
 - **panel_annual_org.csv** : one row per (organization_id, year)

Each includes:
 - **funding** target
 - **topic prevalence** (sum of weights per topic)
 - **deliverable_count**, **publication_count**
 - static org features (country, SME flag, etc.)

In [1]:
# 1) Imports & paths
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().parent  # assumes you're in /notebooks
sys.path.append(str(project_root))

           # adjust to your repo root
data_dir     = project_root / 'data' / 'processed'
agg_dir      = project_root / 'data' / 'aggregated'


# %%
# 2) Load processed tables
#   – projects.csv: for funding & start_date
#   – project_organizations.csv: maps project→organization
#   – project_topic_weights.csv: from Phase 1
#   – deliverables.csv / publications.csv: for counts
#   – organizations.csv: static covariates

projects       = pd.read_csv(data_dir / 'projects.csv', parse_dates=['start_date'])
proj_org       = pd.read_csv(data_dir / 'project_organizations.csv')
ptw_wide       = pd.read_csv(agg_dir  / 'project_topic_weights.csv')
deliverables   = pd.read_csv(data_dir / 'deliverables.csv')
publications   = pd.read_csv(data_dir / 'publications.csv')
organizations  = pd.read_csv(data_dir / 'organizations.csv')

# **Fix datetime parsing for content_update_date**
deliverables['content_update_date'] = pd.to_datetime(
    deliverables['content_update_date'],
    errors='coerce'
)
publications['content_update_date'] = pd.to_datetime(
    publications['content_update_date'],
    errors='coerce'
)

deliverables = deliverables.dropna(subset=['content_update_date'])
publications = publications.dropna(subset=['content_update_date'])

# 3) Enforce consistent dtypes for project_id
for df in (proj_org, ptw_wide, deliverables, publications):
    df['project_id'] = pd.to_numeric(df['project_id'], errors='coerce').astype('Int64')
    
# drop any rows where conversion failed (if any)
proj_org       = proj_org.dropna(subset=['project_id']).astype({'project_id':'int64'})
ptw_wide       = ptw_wide.dropna(subset=['project_id']).astype({'project_id':'int64'})
deliverables   = deliverables.dropna(subset=['project_id']).astype({'project_id':'int64'})
publications   = publications.dropna(subset=['project_id']).astype({'project_id':'int64'})

projects.name       = 'projects'
proj_org.name       = 'project_organizations'
ptw_wide.name       = 'project_topic_weights'
deliverables.name   = 'deliverables'
publications.name   = 'publications'
organizations.name  = 'organizations'
# %%


# print columns of all dataframes : like DF name: col1, col2, ...
for df in [projects, proj_org, ptw_wide, deliverables, publications, organizations]:
    print(f"{df.name}: {', '.join(df.columns)}")

projects: id, acronym, status, title, start_date, end_date, total_cost, ec_max_contribution, ec_signature_date, framework_programme, master_call, sub_call, funding_scheme, nature, objective, content_update_date, rcn, grant_doi, duration_days, duration_months, duration_years, n_institutions, coordinator_name, ec_contribution_per_year, total_cost_per_year, field_class, field, sub_field, niche
project_organizations: project_id, organization_id, role, order_index, ec_contribution, net_ec_contribution, total_cost, end_of_participation, active
project_topic_weights: project_id, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26
deliverables: id, project_id, title, deliverable_type, description, url, collection, content_update_date
publications: id, project_id, title, is_published_as, authors, journal_title, journal_number, published_year, published_pages, issn, isbn, doi, collection, content_update_date
0         organizations
1         organizat

In [2]:
# %%  
# 3) Choose your funding target
#    We’ll forecast `net_ec_contribution`, which is already in proj_org.
#    So just merge in start_date from projects.

proj_org = (
    proj_org
    .merge(
        projects[['id','start_date']],      # only bring the dates
        left_on='project_id',
        right_on='id',
        how='left'
    )
    .rename(columns={'start_date':'proj_start'})
)

ptw_long = (
    ptw_wide
    .melt(
        id_vars=['project_id'],
        var_name='topic_id',
        value_name='weight'
    )
    .astype({'topic_id':'int'})
)

# Now proj_org has proj_start (a Timestamp) and net_ec_contribution

In [3]:
# %%
# 4) Build Quarterly Panel

# 4a) tag each proj_org row with its quarter
proj_org['period_q'] = proj_org['proj_start'].dt.to_period('Q')

# 4b) funding per org×quarter (using net_ec_contribution)
fund_q = (
    proj_org
    .groupby(['organization_id','period_q'])['net_ec_contribution']
    .sum()
    .rename('funding')
)

# 4c) topic prevalence per org×quarter
tp_q = (
    ptw_long
    .merge(
        proj_org[['project_id','organization_id','period_q']],
        on='project_id', how='inner'
    )
    .groupby(['organization_id','period_q','topic_id'])['weight']
    .sum()
    .unstack(fill_value=0)
)

# 4d) deliverable & publication counts
deliv_q = (
    deliverables
    .assign(period_q=lambda df: df.content_update_date.dt.to_period('Q'))
    .merge(proj_org[['project_id','organization_id']], on='project_id')
    .groupby(['organization_id','period_q'])
    .size()
    .rename('deliverable_count')
)

pub_q = (
    publications
    .assign(period_q=lambda df: df.content_update_date.dt.to_period('Q'))
    .merge(proj_org[['project_id','organization_id']], on='project_id')
    .groupby(['organization_id','period_q'])
    .size()
    .rename('publication_count')
)

# 4e) merge all pieces
panel_q = (
    fund_q.to_frame()
    .join(tp_q,    how='left')
    .join(deliv_q, how='left')
    .join(pub_q,   how='left')
    .fillna(0)
    .reset_index()
)

# 4f) add static org features
panel_q = panel_q.merge(
    organizations.drop(columns=['content_update_date']),
    left_on='organization_id',
    right_on='id',
    how='left'
)

# 4g) convert period to timestamp & save
panel_q['ds'] = panel_q['period_q'].dt.to_timestamp('D')
panel_q.to_csv(agg_dir / 'panel_quarterly_org.csv', index=False)
print("✅ Saved quarterly panel:", agg_dir/'panel_quarterly_org.csv')

✅ Saved quarterly panel: c:\Users\suley\Desktop\ManaMa\MDA\EU_Horizon_Dashboard\data\aggregated\panel_quarterly_org.csv


In [4]:
# %%
# 5) Build Annual Panel

# 5a) tag each proj_org row with its year
proj_org['period_y'] = proj_org['proj_start'].dt.to_period('Y')

# 5b) funding per org×year
fund_y = (
    proj_org
    .groupby(['organization_id','period_y'])['net_ec_contribution']
    .sum()
    .rename('funding')
)

# 5c) topic prevalence per org×year
tp_y = (
    ptw_long
    .merge(
        proj_org[['project_id','organization_id','period_y']],
        on='project_id', how='inner'
    )
    .groupby(['organization_id','period_y','topic_id'])['weight']
    .sum()
    .unstack(fill_value=0)
)

# 5d) deliverable & publication counts per org×year
deliv_y = (
    deliverables
    .assign(period_y=lambda df: df.content_update_date.dt.to_period('Y'))
    .merge(proj_org[['project_id','organization_id']], on='project_id', how='inner')
    .groupby(['organization_id','period_y'])
    .size()
    .rename('deliverable_count')
)

pub_y = (
    publications
    .assign(period_y=lambda df: df.content_update_date.dt.to_period('Y'))
    .merge(proj_org[['project_id','organization_id']], on='project_id', how='inner')
    .groupby(['organization_id','period_y'])
    .size()
    .rename('publication_count')
)

# 5e) merge all pieces & fill missing
panel_y = (
    fund_y.to_frame()
    .join(tp_y,    how='left')
    .join(deliv_y, how='left')
    .join(pub_y,   how='left')
    .fillna(0)
    .reset_index()
)

# 5f) add static org features
panel_y = panel_y.merge(
    organizations.drop(columns=['content_update_date']),
    left_on='organization_id',
    right_on='id',
    how='left'
)

# 5g) convert period to timestamp & save
panel_y['ds'] = panel_y['period_y'].dt.to_timestamp('D')
panel_y.to_csv(agg_dir / 'panel_annual_org.csv', index=False)
print("✅ Saved annual panel:  ", agg_dir/'panel_annual_org.csv')


✅ Saved annual panel:   c:\Users\suley\Desktop\ManaMa\MDA\EU_Horizon_Dashboard\data\aggregated\panel_annual_org.csv


In [5]:
# print some data from panel quarterly
print("\nSample from quarterly panel:")
print(panel_q.head(30).to_string(index=False))


Sample from quarterly panel:
 organization_id period_q   funding             0             1             2             3             4             5             6             7             8             9            10            11            12            13            14            15            16            17            18            19            20            21            22            23            24            25            26  deliverable_count  publication_count        id          name              short_name    vat_number   sme activity_type                 street post_code                     city country nuts_code geolocation organization_url                                                                                                   contact_form         ds
       873166065   2024Q4      0.00  1.409723e-01  4.590141e-01  3.230876e-02  3.097276e-02  2.665145e-02  3.236972e-02  1.543698e-02  1.059682e-02  1.586988e-02  1.265273e-02  1.467227e-02  1.177519e-02  1.18